# Ep. 04 — Llamada a herramientas (tool calling)

Solo, el modelo solo genera texto a partir del prompt. Con **tools**, puede pedir que tu código ejecute una función (clima, pedido, base de datos) y use el resultado en la respuesta final.


## Limitación — sin tools, el modelo solo "habla"

Sin herramientas registradas, el asistente **no busca datos externos**. Pregunta la temperatura ahora: inventa, se niega o usa conocimiento estático del entrenamiento — no hay API de clima en el loop.


In [ ]:
import os
import json
import urllib.request

API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
API_URL = "https://api.openai.com/v1/chat/completions"

def ask(messages):
    """messages: string O lista de dicts {role, content}."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    if not API_KEY:
        raise SystemExit("Define OPENAI_API_KEY y vuelve a ejecutar.")
    body = json.dumps({"model": MODEL, "messages": messages}).encode()
    req = urllib.request.Request(
        API_URL,
        data=body,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode())
    return data["choices"][0]["message"]["content"]

# Sin tools: el modelo solo tiene el prompt — no hay clima en vivo.
pregunta = "¿Cuál es la temperatura ahora en São Paulo?"
respuesta = ask(pregunta)
print(respuesta)


## Definir una tool (JSON Schema, estilo OpenAI)

Cada tool es un objeto `type: function` con `name`, `description` y `parameters` (JSON Schema). El modelo **elige** si llama y con qué argumentos; **tu código** ejecuta.


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Devuelve el clima actual (datos locales simulados) para una ciudad.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Nombre de la ciudad, ej.: São Paulo",
                    }
                },
                "required": ["city"],
            },
        },
    }
]

# Datos deterministas — el cuerpo de la tool corre offline; el LLM aún necesita la API.
WEATHER = {
    "são paulo": {"temp_c": 23, "condition": "parcialmente nublado", "humidity": 68},
    "sao paulo": {"temp_c": 23, "condition": "parcialmente nublado", "humidity": 68},
    "lisboa": {"temp_c": 19, "condition": "soleado", "humidity": 55},
    "madrid": {"temp_c": 27, "condition": "cielo despejado", "humidity": 40},
}

def get_weather(city: str) -> str:
    key = city.strip().lower()
    data = WEATHER.get(key)
    if not data:
        data = {
            "temp_c": None,
            "condition": "desconocido",
            "humidity": None,
            "note": f"sin datos para {city}",
        }
    return json.dumps({"city": city, **data}, ensure_ascii=False)

FUNCTIONS = {
    "get_weather": get_weather,
}

print("TOOLS:", json.dumps(TOOLS, indent=2, ensure_ascii=False))
print("demo get_weather:", get_weather("São Paulo"))


## Registrar tools y verificar `tool_calls`

Envía `tools=[...]` y `tool_choice="auto"` en el body. Después de cada respuesta:

1. Inspecciona `message.tool_calls`.
2. **Solo si** la lista no está vacía → invoca las funciones locales, append `role: tool`, llama de nuevo.
3. Si no hay `tool_calls` → usa el `content` del asistente (no invoca nada).


In [ ]:
def chat_completion(messages, tools=None):
    """Una llamada a la API; devuelve el objeto message completo (puede tener tool_calls)."""
    if not API_KEY:
        raise SystemExit("Define OPENAI_API_KEY y vuelve a ejecutar.")
    payload = {"model": MODEL, "messages": messages}
    if tools is not None:
        payload["tools"] = tools
        payload["tool_choice"] = "auto"
    body = json.dumps(payload).encode()
    req = urllib.request.Request(
        API_URL,
        data=body,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode())
    return data["choices"][0]["message"]

def ask_with_tools(messages, tools=TOOLS, functions=FUNCTIONS, max_rounds=3):
    """
    Loop con verificación explícita:
      1) llama a la API con tools registradas
      2) inspecciona message.tool_calls
      3) SOLO si hay tool_calls → ejecuta, append role=tool, llama de nuevo
      4) si no hay → usa el content normal (no invoca nada)
    """
    messages = list(messages)
    for round_i in range(max_rounds):
        msg = chat_completion(messages, tools=tools)

        # --- verificación explícita ---
        tool_calls = msg.get("tool_calls") or []
        has_tool_calls = isinstance(tool_calls, list) and len(tool_calls) > 0
        print(f"[round {round_i}] ¿tool_calls presentes? {has_tool_calls} (n={len(tool_calls)})")

        assistant = {"role": "assistant", "content": msg.get("content")}
        if has_tool_calls:
            assistant["tool_calls"] = tool_calls
        messages.append(assistant)

        if not has_tool_calls:
            # Sin tool_calls: no invoca funciones; devuelve el texto del asistente.
            content = msg.get("content") or ""
            print("Ninguna tool solicitada — usando content del asistente.")
            return content

        # Solo llega aquí si tool_calls es una lista no vacía.
        for tc in tool_calls:
            name = tc["function"]["name"]
            args = json.loads(tc["function"]["arguments"] or "{}")
            print(f"→ invocar: {name}({args})")
            result = functions[name](**args)
            print(f"← resultado: {result}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc["id"],
                "content": result,
            })
        # siguiente iteración: segunda (o N-ésima) completion con los resultados

    raise RuntimeError("max_rounds alcanzado sin respuesta final")

mensajes = [
    {
        "role": "system",
        "content": "Eres un asistente. Usa get_weather cuando necesites el clima actual.",
    },
    {"role": "user", "content": "¿Cuál es la temperatura ahora en São Paulo?"},
]
final = ask_with_tools(mensajes)
print("---")
print(final)


## Takeaways

1. Sin tools, el LLM solo completa texto desde el contexto — no hay side effects.
2. Tools = schema en el request + función local + mensaje `role: tool` con el resultado.
3. Siempre **verifica** `tool_calls` antes de invocar; si está vacío, devuelve el `content` normal.
